In [1]:
from akula import API_TOKEN
#print(API_TOKEN)

In [ ]:
import telebot
from telebot import types
import sqlite3
from datetime import datetime
import random

user_data = {}
movie_id = None
bot = telebot.TeleBot(API_TOKEN)


# Проверка, существует ли пользователь в базе данных
def user_exists(user_id):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    cursor.execute('''SELECT COUNT(1) FROM users WHERE user_id = ?''', (user_id,))
    exists = cursor.fetchone()[0] > 0
    conn.close()
    return exists

# Сохранение информации о пользователе в базу данных
def save_user_info(user_info):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    if user_exists(user_info['user_id']):
        cursor.execute('''UPDATE users
                          SET first_name = ?, last_name = ?, username = ?, language_code = ?, is_bot = ?, birth_date = ?, last_activity_date = ?
                          WHERE user_id = ?''',
                       (user_info['first_name'], user_info.get('last_name'), user_info.get('username'),
                        user_info.get('language_code'), user_info['is_bot'], user_info.get('birth_date'),
                        user_info['last_activity_date'], user_info['user_id']))
    else:
        cursor.execute('''INSERT INTO users
                          (user_id, first_name, last_name, username, language_code, is_bot, birth_date, registration_date, last_activity_date)
                          VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)''',
                       (user_info['user_id'], user_info['first_name'], user_info.get('last_name'), user_info.get('username'),
                        user_info.get('language_code'), user_info['is_bot'], user_info.get('birth_date'),
                        user_info['registration_date'], user_info['last_activity_date']))
    conn.commit()
    conn.close()

# Обновление даты последней активности пользователя
def update_last_activity(user_id):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    cursor.execute('''UPDATE users 
                      SET last_activity_date = ? 
                      WHERE user_id = ?''',
                   (datetime.now().strftime('%Y-%m-%d %H:%M:%S'), user_id))
    conn.commit()
    conn.close()

# Получение случайного фильма из базы данных, который еще не был оценен пользователем
def get_random_movie(user_id):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()

    # Получаем список ID фильмов, которые уже были оценены пользователем
    cursor.execute('''SELECT movie_id FROM actions WHERE user_id = ?''', (user_id,))
    rated_movies = [row[0] for row in cursor.fetchall()]

    # Получаем список всех фильмов
    cursor.execute('''SELECT id FROM movies''')
    all_movies = [row[0] for row in cursor.fetchall()]

    # Проверяем, есть ли еще неоцененные фильмы
    unrated_movies = set(all_movies) - set(rated_movies)
    if not unrated_movies:
        # Если все фильмы уже оценены, возвращаем None
        return None

    # Получаем случайный фильм, который еще не был оценен пользователем
    movie_id = random.choice(list(unrated_movies))
    cursor.execute('''SELECT id, name, slogan, description, year
                      FROM movies
                      WHERE id = ?''', (movie_id,))
    movie = cursor.fetchone()
    conn.close()
    return movie



# Получение URL-адреса превью по ID фильма
def get_preview_url(movie_id):
    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()
    cursor.execute('''SELECT preview_url FROM posters WHERE movie_id = ?''', (movie_id,))
    preview_url = cursor.fetchone()
    conn.close()
    return preview_url[0] if preview_url else None


# Обработчик команды /start
@bot.message_handler(commands=['start'])
def send_welcome(message):
    user = message.from_user
    user_info = {
        'user_id': user.id,
        'first_name': user.first_name,
        'last_name': user.last_name,
        'username': user.username,
        'language_code': user.language_code,
        'is_bot': user.is_bot,
        'birth_date': None,
        'registration_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'last_activity_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    }
    if user_exists(user.id):
        # Если пользователь уже существует, обновляем только дату последней активности
        update_last_activity(user.id)
    else:
        # Если пользователь новый, сохраняем всю информацию
        save_user_info(user_info)
    
    bot.reply_to(message, f"Привет, {user.first_name}! Добро пожаловать в наш бот для оценки фильмов.")
    send_random_movie(message)



# Отправка случайного фильма для оценки
def send_random_movie(message):
    global movie_id
    user = message.from_user
    movie = get_random_movie(user.id)
    if movie:
        movie_id, title, tagline, description, release_year = movie

        preview_url = get_preview_url(movie_id)

        # Создание кнопок для оценки фильма
        reply_markup = types.ReplyKeyboardMarkup(row_width=3, resize_keyboard=True)
        btn_dislike = types.KeyboardButton('👎')
        btn_menu = types.KeyboardButton('📺')
        btn_like = types.KeyboardButton('👍')
        reply_markup.add(btn_dislike, btn_menu, btn_like)

        # Формирование текста сообщения
        movie_text = f"*{title}* \n\n"
        if tagline:
            movie_text += f"*{tagline}*\n\n"
        if description:
            movie_text += f"{description}\n\n"
        movie_text += f"*Год фильма: {release_year}*"

        if preview_url:
            # Отправка фотографии с inline_markup
            bot.send_photo(message.chat.id, preview_url, caption=movie_text, parse_mode='Markdown', reply_markup=reply_markup)
        else:
            bot.send_message(message.chat.id, "Не удалось найти превью для фильма.")
    else:
        bot.send_message(message.chat.id, "Вы оценили все фильмы. Спасибо за ваше участие!")



#Обработка '👎', '👍'
@bot.message_handler(func=lambda message: message.text in ['👎', '👍'])
def movie_rating_handler(message):
    user = message.from_user
    update_last_activity(user.id)  # Обновляем дату последней активности

    rating = None
    want_to_watch = None

    if message.text == '👎':
        want_to_watch = -1
        bot.reply_to(message, "Вы поставили отрицательную оценку.")
    elif message.text == '👍':
        want_to_watch = 1
        bot.reply_to(message, "Вы поставили положительную оценку.")

    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()

    cursor.execute('''INSERT INTO actions
                      (user_id, movie_id, want_to_watch, rating)
                      VALUES (?, ?, ?, ?)''',
                   (user.id, movie_id, want_to_watch, rating))

    conn.commit()
    conn.close()

    # Отправляем следующий фильм после оценки
    send_random_movie(message)


#Обработка '📺'
@bot.message_handler(func=lambda message: message.text == '📺')
def show_liked_movies(message):
    global is_selecting_movie
    user = message.from_user
    update_last_activity(user.id)  # Обновляем дату последней активности

    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()

    # Получаем список фильмов, на которые пользователь поставил "👍"
    cursor.execute('''SELECT movies.name, movies.year
                      FROM movies
                      JOIN actions ON movies.id = actions.movie_id
                      WHERE actions.user_id = ? AND actions.want_to_watch = 1''',
                   (user.id,))

    liked_movies = cursor.fetchall()
    conn.close()

    if liked_movies:
        # Формируем список фильмов в виде строки
        movies_list = "\n".join([f" `{title}` ({year})" for title, year in liked_movies])
        bot.reply_to(message, f"Список фильмов, на которые вы поставили 👍 :\n\n{movies_list}\n\nПожалуйста, введите название фильма, чтобы узнать площадки для просмотра.", parse_mode="Markdown")
        is_selecting_movie = True
        bot.register_next_step_handler(message, show_watchability)
    else:
        bot.reply_to(message, "Вы еще не поставили 👍 ни одному фильму.")
        send_random_movie(message)




def show_watchability(message):
    movie_name = message.text

    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()

    # Получаем информацию о фильме по названию
    cursor.execute('''SELECT id, name, year
                      FROM movies
                      WHERE name LIKE ?''',
                   (f"%{movie_name}%",))

    movie = cursor.fetchone()

    if movie:
        movie_id, title, year = movie

        # Получаем информацию о площадке для просмотра фильма
        cursor.execute('''SELECT service_name, link
                          FROM watchability
                          WHERE movie_id = ?''',
                       (movie_id,))

        watchability = cursor.fetchall()

        if watchability:
            # Format the platforms list as a string
            platforms_list = "\n".join([f"`{service_name[0]}`" for service_name in watchability])
    
            # Create a "Back" button
            back_button = types.KeyboardButton("Back")
    
            # Create a keyboard with the platforms list and the "Back" button
            keyboard = types.ReplyKeyboardMarkup(resize_keyboard=True)
            keyboard.add(*[types.KeyboardButton(platform[0]) for platform in watchability])
            keyboard.add(back_button)
    
            # Send the platforms list to the user with the keyboard
            bot.reply_to(message, f"Площадки для просмотра фильма '{title}' ({year}):\n\n{platforms_list}\n\nПожалуйста, выберите площадку для просмотра.", reply_markup=keyboard, parse_mode="Markdown")
    
            # Save the watchability list in the user's context
            user_data[message.from_user.id] = watchability
    
            # Register the platform selection handler
            bot.register_next_step_handler(message, handle_platform_selection)
        else:
            bot.reply_to(message, f"К сожалению, не удалось найти площадки для просмотра фильма '{title}' ({year}). Пожалуйста, выберите другой фильм.")
            show_liked_movies(message)
    else:
        # Создаем клавиатуру с кнопками "Выбрать другой фильм" и "Поискать новый фильм"
        markup = types.ReplyKeyboardMarkup(row_width=1, resize_keyboard=True)
        btn_search_new = types.KeyboardButton('Поискать новый фильм')
        markup.add(btn_search_new)

        bot.reply_to(message, f"К сожалению, не удалось найти фильм с названием '{movie_name}'.", reply_markup=markup)
        bot.register_next_step_handler(message, handle_movie_selection_buttons)


    conn.close()

def handle_movie_selection_buttons(message):
    if message.text == 'Поискать новый фильм':
        send_random_movie(message)
    
def handle_platform_selection(message):
    user_id = message.from_user.id
    platform_name = message.text

    # Check if the user typed "Back" to go back to the previous step
    if platform_name == "Back":
        # Display the platform selection step again
        show_watchability(message)
        return

    # Check if the user's message is a valid platform name
    if platform_name in [platform[0] for platform in user_data[user_id]]:
        # Retrieve the corresponding link from the watchability list
        link = [platform[1] for platform in user_data[user_id] if platform[0] == platform_name][0]

        # Send the link to the user
        bot.reply_to(message, f"Ссылка на площадку для просмотра фильма: {link}")

        # Display a message that instructs the user to type "Back" to go back to the previous step
        bot.send_message(message.chat.id, "Чтобы выбрать другую площадку для просмотра, напишите 'Back'.")
    else:
        # Prompt the user to enter a valid platform name or type "Back" to go back to the previous step
        bot.reply_to(message, "Некорректный ввод. Пожалуйста, выберите площадку для просмотра из списка или напишите 'Back' для выбора другого фильма.")

        # Register the platform selection handler for the next step
        bot.register_next_step_handler(message, handle_platform_selection)


#Обрабочик /drop
@bot.message_handler(commands=['drop'])
def drop_user_data(message):
    user = message.from_user

    conn = sqlite3.connect('movies.db')
    cursor = conn.cursor()

    # Удаляем все записи пользователя из таблицы "actions"
    cursor.execute('''DELETE FROM actions WHERE user_id = ?''', (user.id,))

    conn.commit()
    conn.close()

    bot.reply_to(message, "Ваши сохраненные списки оценок успешно очищены.")
    send_random_movie(message)
    
# Обработчик некорректного ввода
@bot.message_handler(func=lambda message: message.text not in ['👎', '📺', '👍'])
def handle_incorrect_input(message):
    bot.reply_to(message, "Некорректный ввод. Пожалуйста, используйте кнопки для оценки фильма.")

if __name__ == '__main__':
    bot.polling(none_stop=True)
